# Hidden CKD

#### Features
The following are the features used in this study in order of appearance
- Date of event: Date of the screening
- Gender: Gender of the patient (M: Male, F: Female)
- Ethnicity: The ethnicity of the participant
- Age: Age of the patient (years)
- Height (cm): Height of the participant in cm
- Weight (kg): Weight of the participant in kg
- BMI: BMI of the participant
- BMI Category: Classification of the particpant BMI according to NICE guidelines
- Systolic, Diastolic: The systolic and diastolic of the partcipants
- Pulse Pressure: 
- BP Category: Classification of the particpant BP according to NICE guidelines
- Has High BP:
- Has Diabetes
- Has Kidney Disease
- On BP Medication?
- On Diabetes Medication?
- On Cholesterol Medication?
- On Other Medication?
- Family History of Kidney Disease: Whether the patient has a family history of kidney disease (Definitely Yes, Definitely Not, Not Sure)
- uACR: Urine albumin to creatinine ratio of the participants as measured using a urine dipstick (Normal, Abnormal, High Abnormal)
- CKD Risk: A calculation of CKD risk by combining research findings

This is how CKD risk is calculated

low risk = sys<140, dia<90, uACR='Normal', Has_Diabetes=False or Family_KD=False<br>
moderate risk = sys<140, dia<90, uACR='Abnormal', Has_Diabetes=True or Family_KD=True<br>
high risk = sys>=180, dia>=120, uACR='High Abnormal', Has_Diabetes=False or Family_KD=False<br>

In [1]:
import numpy as np
import pandas as pd
from src.config import PROCESSED_DATA_DIR

2025-08-05 19:18:23.773 | INFO     | src.config:<module>:11 - PROJ_ROOT path is: /Users/Edward/Documents/GitHub/hidden-ckd


In [2]:
filename = PROCESSED_DATA_DIR / 'hidden_ckd_processed.csv'
df = pd.read_csv(filename)
df.head()

,Date,Gender,Ethnicity,S_Ethnicity,Ethnicity_Black,DOB,Age,Age_Category,Height,Weight,...,Has_Diabetes,Has_KD,Has_HD,BP_Meds,Diabetes_Meds,Cholesterol_Meds,Other_Meds,Family_KD,uACR,CKD_Risk
0,23/10/2022,Male,Black Caribbean,Black,True,21/05/1946,76.5,>70,161.0,64.0,...,False,False,False,False,False,False,False,Definitely not,Abnormal,Moderate
1,23/10/2022,Male,Black African (West Africa),Black,True,25/01/1970,52.8,41-55,163.0,78.0,...,False,False,False,False,False,False,False,Definitely not,Abnormal,Moderate
2,26/08/2023,Male,Black Caribbean,Black,True,14/07/2005,18.1,<25,167.0,91.0,...,False,False,False,False,False,False,False,Definitely not,Normal,Low
3,28/04/2023,Male,Black Caribbean,Black,True,25/04/1969,54.0,41-55,168.0,87.0,...,False,False,False,False,False,False,False,Definitely not,Abnormal,Moderate
4,06/11/2022,Female,Black African (West Africa),Black,True,03/11/1979,43.0,41-55,187.0,109.0,...,False,False,False,False,False,False,False,Definitely not,Abnormal,Moderate


## ML Models (without Using Blood Pressure Readings)

In [3]:
# Import modules
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import PowerTransformer
from sklearn.impute import SimpleImputer

In [4]:
from src.utils.model_utils import (resample_data, stratified_split, random_split,
                                   load_model, evaluate_model)
from src.utils.models import (train_ensemble1, train_ensemble2, train_gnb,
                              train_knn, train_lr, train_rf, train_xg, train_regressor)
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

In [5]:
from imblearn.over_sampling import SMOTENC

num_features = ['Age', 'Height', 'Weight', 'Systolic', 'Diastolic']
nom_features = ['Family_KD', 'uACR']
ord_features = ['Has_Hpt', 'Has_Diabetes']

X = df[num_features + nom_features + ord_features]
y = df['CKD_Risk']

resampler = SMOTENC(sampling_strategy='not majority', random_state=42, categorical_features=[5, 6])
X_res, y_res = resampler.fit_resample(X, y)

In [6]:
# Creating preprocessing pipelines for both numeric and nominal and ordinal data.
num_features = ['Age', 'Height', 'Weight']
num_transformer = Pipeline(steps=[
    ('scaler', PowerTransformer(method='yeo-johnson'))])

nom_features = ['Family_KD']
nom_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ])

ord_features = ['Has_Hpt', 'Has_Diabetes']
ord_categories = [[False, True] for _ in ord_features]
ord_transformer = Pipeline(steps=[
    ('ord_enc', OrdinalEncoder(categories=ord_categories))
])

# Combining everything into one preprocessor
preprocessor_reg = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('nom', nom_transformer, nom_features),
        ('ord', ord_transformer, ord_features)])

# Define regression features and targets
X = X_res[num_features + nom_features + ord_features]
y = X_res[['Systolic', 'Diastolic']]

# Function to fit transform preprocessor
def preprocess_x(preprocessor, X_train, X_test):
    preprocessor.fit(X_train)
    X_train = preprocessor.transform(X_train)
    X_test = preprocessor.transform(X_test)
    return X_train, X_test

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_processed, X_test_processed = preprocess_x(preprocessor_reg, X_train, X_test)

# Train regressor
model_filename = 'bp_regressor.pkl'
regressor = train_regressor(X_train_processed, y_train, model_filename)

y_pred = regressor.predict(X_test_processed)
mse = mean_squared_error(y_test, y_pred, multioutput='raw_values')
print(f'Tuned MSE for Systolic: {mse[0]:.2f}')
print(f'Tuned MSE for Diastolic: {mse[1]:.2f}')


Fitting 10 folds for each of 720 candidates, totalling 7200 fits
Best hyperparameters: {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 8, 'n_estimators': 100, 'subsample': 0.8}
Tuned MSE for Systolic: 226.47
Tuned MSE for Diastolic: 89.95


In [8]:
df[['Systolic', 'Diastolic']] = pd.DataFrame(regressor.predict(preprocessor_reg.transform(X)))

# Creating preprocessing pipelines for both numeric and nominal and ordinal data.
num_features = ['Age', 'Height', 'Weight', 'Systolic', 'Diastolic']
num_transformer = Pipeline(steps=[
    ('power_transform', PowerTransformer(method='yeo-johnson'))])

nom_features = ['S_Ethnicity', 'Family_KD', 'Gender']
nom_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ])

ord_features = ['Has_Hpt', 'Has_Diabetes']
ord_categories = [[False, True] for _ in ord_features]
ord_transformer = Pipeline(steps=[
    ('ord_enc', OrdinalEncoder(categories=ord_categories))
])

# Combining everything into one preprocessor
preprocessor_clf = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('nom', nom_transformer, nom_features),
        ('ord', ord_transformer, ord_features)])

# Import processed data
X = df[num_features + nom_features + ord_features]

In [9]:
X = preprocessor_clf.fit_transform(X)
X = pd.DataFrame(data=X)
y = df['CKD_Risk']
target_mapping = {'Low': 0, 'Moderate': 1, 'High': 2}

# Correct class imbalance
X_over, y_over = resample_data(X, y, target_mapping)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = stratified_split(X_over, y_over)

In [12]:
# Import modules
from src.utils.model_utils import load_model, evaluate_model

# Load test data
target_mapping = {'Low': 0, 'Moderate': 1, 'High': 2}

# Load model
model_filename = 'classifier_bp.pkl'
model = load_model(model_filename)

# Evaluate model
evaluate_model(model, X_test, y_test, target_mapping)

ValueError: Feature shape mismatch, expected: 16, got 18

In [ ]:
data = pd.DataFrame([{
    'Age': 66,
    'Height': 154,
    'Weight': 67.3,
    'Systolic': 120,
    'Diastolic': 90,
    'S_Ethnicity': 'Black',
    'Family_KD': 'Definitely not',
    'Gender': 'Male',
    'Has_Hpt': True,
    'Has_Diabetes': True,
}])

# Run pipeline
full_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('model', model)
])

# Prediction
prediction = full_pipeline.predict(data)
print("Prediction:", prediction)

probabilities = full_pipeline.predict_proba(data)
for i, probs in enumerate(probabilities):
    print(f"Sample {i}:")
    for cls, prob in enumerate(probs):
        print(f"  Class {cls} probability: {prob:.4f}")

NameError: name 'preprocessor' is not defined